# 03. 학습(Train) 심화 — 인자·하이퍼파라미터·로그 읽기

배우는 것
1. 학습 인자 총정리 (무엇을 왜 조절하나)
2. 실제 학습 실행 & 출력 로그 해석
3. `results.csv`로 학습 곡선 그리기
4. 저장물(weights, plots) 이해
5. 이어학습(resume) / 재개

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
import torch
from ultralytics import YOLO

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", DEVICE)

## 1. 학습 인자 총정리

| 그룹 | 인자 | 의미 |
|---|---|---|
| **필수** | `data` | 데이터셋 YAML |
| | `epochs` | 전체 데이터 반복 횟수 |
| | `imgsz` | 입력 크기 (기본 640) |
| | `batch` | 배치 크기 (`-1`이면 자동) |
| | `device` | `mps` / `cpu` / `0`(cuda) |
| **최적화** | `lr0` | 초기 학습률 |
| | `optimizer` | `auto`/`SGD`/`Adam`/`AdamW` |
| | `weight_decay` | 가중치 규제 |
| | `warmup_epochs` | 워밍업 구간 |
| **일반화** | `patience` | 개선 없을 때 조기종료 대기 |
| | `dropout` | 드롭아웃 |
| | `augment`/`hsv_*`/`fliplr`/`mosaic` | 데이터 증강 |
| **관리** | `project`/`name` | 결과 저장 위치 |
| | `resume` | 중단된 학습 이어하기 |
| | `save_period` | N epoch마다 체크포인트 저장 |

> 전이학습(사전학습 .pt 로드)이 기본이라, 소량 데이터면 `epochs`를 크게 잡고 `patience`로 조기종료를 거는 전략이 흔합니다.

## 2. 실제 학습 실행

coco8로 짧게 돌립니다. 출력되는 로그를 아래에서 해석해요.

In [ ]:
model = YOLO("yolo26n.pt")

res = model.train(
    data="coco8.yaml",
    epochs=20,
    imgsz=640,
    batch=16,
    device=DEVICE,
    workers=0,
    patience=10,      # 10 epoch 개선 없으면 조기종료
    name="train_deep",
    exist_ok=True,
)
print("저장 폴더:", res.save_dir)

### 로그 읽는 법
학습 중 한 줄은 대략 이렇게 생겼습니다:
```
Epoch  GPU_mem  box_loss  cls_loss  dfl_loss  Instances  Size
 5/20   2.3G     0.98      1.67      0.011      13        640
```
- **box_loss**: 박스 위치 오차 (내려갈수록 좋음)
- **cls_loss**: 클래스 분류 오차
- **dfl_loss**: 박스 경계 분포 오차 (distribution focal loss)
- 각 epoch 뒤에 val 지표(P, R, mAP50, mAP50-95)가 함께 찍힙니다.

→ **loss는 내려가고 mAP는 올라가면** 학습이 잘 되는 것.

## 3. 학습 곡선 직접 그리기 (results.csv)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(f"{res.save_dir}/results.csv")
df.columns = [c.strip() for c in df.columns]

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(df["epoch"], df["train/box_loss"], label="box_loss")
ax[0].plot(df["epoch"], df["train/cls_loss"], label="cls_loss")
ax[0].set_title("train loss"); ax[0].set_xlabel("epoch"); ax[0].legend()

ax[1].plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP50")
ax[1].plot(df["epoch"], df["metrics/mAP50-95(B)"], label="mAP50-95")
ax[1].set_title("val mAP"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

## 4. 저장물 이해

`save_dir` 안에 자동 생성되는 것들:
- `weights/best.pt` — val 성능이 가장 좋았던 가중치 (**보통 이걸 씀**)
- `weights/last.pt` — 마지막 epoch 가중치 (이어학습용)
- `results.csv` / `results.png` — 지표 로그·곡선
- `confusion_matrix.png`, `*_curve.png` — 성능 시각화
- `train_batch*.jpg` — 증강이 적용된 학습 배치 미리보기
- `args.yaml` — 이 학습에 쓰인 모든 설정

In [ ]:
from pathlib import Path
for p in sorted(Path(res.save_dir).glob("*")):
    print("  ", p.name)
print("weights/:")
for p in sorted((Path(res.save_dir) / "weights").glob("*")):
    print("  ", p.name)

## 5. 이어학습 & 저장 가중치 재사용

In [ ]:
# 방금 학습한 best.pt 를 다시 불러 추론에 쓰기
best = YOLO(f"{res.save_dir}/weights/best.pt")
out = best.predict("https://ultralytics.com/images/bus.jpg", device=DEVICE, verbose=False)
print("내 학습 모델 탐지 수:", len(out[0].boxes))

# 중단된 학습을 이어서 하려면:
#   m = YOLO(f"{res.save_dir}/weights/last.pt")
#   m.train(resume=True)

## 6. 연습문제
1. `lr0=0.001`, `optimizer="AdamW"`로 다시 학습해 mAP를 비교해 보세요.
2. `mosaic=0.0`으로 증강을 끄고 `train_batch0.jpg`가 어떻게 달라지는지 확인해 보세요.
3. `results.csv`에서 mAP50-95가 최고였던 epoch을 찾아 출력해 보세요.

**다음 노트북 →** `04_validation_metrics` : 지표 해석

In [ ]:
# 연습 공간
